# 호출 한도와 재시도·폴백

호출 한도, 재시도, 폴백은 서로 다른 실패 조건을 어떻게 나누어 처리할까요?

## 실행 환경 준비

실습 파일은 `labs/part6/ch22-middleware/notebooks/04-호출-한도와-재시도-폴백.ipynb`입니다. Python 3.12 새 커널에서 위에서부터 실행합니다.

## 패키지 버전 확인

In [1]:
# 이 셀은 패키지 버전 확인 작업을 수행하고 확인할 값을 출력합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langgraph==1.2.11" "python-dotenv==1.2.3"
from importlib.metadata import version
print("langchain", version("langchain"))
print("langchain-openai", version("langchain-openai"))
print("langgraph", version("langgraph"))

Note: you may need to restart the kernel to use updated packages.
langchain 1.4.0
langchain-openai 1.6.0
langgraph 1.2.11


## 결정적 모델 준비

In [2]:
# 이 셀은 결정적 모델 준비 작업을 수행하고 확인할 값을 출력합니다.
# 정해 둔 응답으로 미들웨어 실행 순서를 재현합니다.
from typing import Any
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.tools import tool
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
class ScriptedModel(BaseChatModel):
    replies: list[AIMessage]
    index: int = 0
    @property
    def _llm_type(self) -> str: return "scripted-model"
    def _generate(self, messages: Any, stop: Any = None, run_manager: Any = None, **kwargs: Any) -> ChatResult:
        reply = self.replies[min(self.index, len(self.replies) - 1)]
        self.index += 1
        return ChatResult(generations=[ChatGeneration(message=reply)])
    def bind_tools(self, tools: Any, *, tool_choice: Any = None, **kwargs: Any) -> "ScriptedModel":
        return self
@tool
def lookup_hours(day: str) -> str:
    """자료실 운영 시간을 요일로 조회합니다."""
    return "평일 09:00~20:00" if day == "평일" else "토요일 10:00~17:00"

## 재시도 모델 준비

In [3]:
# 이 셀은 재시도 모델 준비 작업을 수행하고 확인할 값을 출력합니다.
from langchain.agents.middleware import ModelRetryMiddleware, ModelFallbackMiddleware, ModelCallLimitMiddleware
class FlakyModel(ScriptedModel):
    failures: int = 1
    calls: int = 0
    def _generate(self, messages: Any, stop: Any = None, run_manager: Any = None, **kwargs: Any) -> ChatResult:
        self.calls += 1
        if self.calls <= self.failures: raise TimeoutError("temporary")
        return super()._generate(messages,stop,run_manager,**kwargs)
flaky=FlakyModel(replies=[AIMessage("재시도 성공")],failures=1)

## 재시도 실험

In [4]:
# 이 셀은 재시도 실험 작업을 수행하고 확인할 값을 출력합니다.
retry_agent=create_agent(flaky,tools=[],middleware=[ModelRetryMiddleware(max_retries=2,retry_on=(TimeoutError,),initial_delay=0,jitter=False)])
retry_result=retry_agent.invoke({"messages":[{"role":"user","content":"상태 확인"}]})
print("calls:",flaky.calls,"answer:",retry_result["messages"][-1].text)
assert flaky.calls==2

calls: 2 answer: 재시도 성공


## 폴백 실험

In [5]:
# 이 셀은 폴백 실험 작업을 수행하고 확인할 값을 출력합니다.
primary=FlakyModel(replies=[AIMessage("주 모델")],failures=5)
backup=ScriptedModel(replies=[AIMessage("보조 모델 응답")])
fallback_agent=create_agent(primary,tools=[],middleware=[ModelFallbackMiddleware(primary,backup)])
fallback_result=fallback_agent.invoke({"messages":[{"role":"user","content":"상태 확인"}]})
print("answer:",fallback_result["messages"][-1].text)
assert fallback_result["messages"][-1].text=="보조 모델 응답"

answer: 보조 모델 응답


## 호출 한도 실제 차단

In [6]:
# 이 셀은 호출 한도 실제 차단 작업을 수행하고 확인할 값을 출력합니다.
limit=ModelCallLimitMiddleware(run_limit=1,exit_behavior="end")
limited_model=ScriptedModel(replies=[
 AIMessage("",tool_calls=[{"name":"lookup_hours","args":{"day":"평일"},"id":"limit_1","type":"tool_call"}]),
 AIMessage("두 번째 모델 응답")])
limit_agent=create_agent(limited_model,tools=[lookup_hours],middleware=[limit])
limit_result=limit_agent.invoke({"messages":[{"role":"user","content":"평일 운영 시간은?"}]})
print("model_calls:",limited_model.index)
print("last_message:",limit_result["messages"][-1].text)
assert limited_model.index==1 and "limit" in limit_result["messages"][-1].text.lower()

model_calls: 1
last_message: Model call limits exceeded: run limit (1/1)


## 정책별 결과 표

In [7]:
# 이 셀은 정책별 결과 표 작업을 수행하고 확인할 값을 출력합니다.
rows={"retry_calls":flaky.calls,"fallback_primary_calls":primary.calls,"limit":limit.run_limit,"limited_model_calls":limited_model.index,"estimated_cost_usd":0.0}
print(rows)
assert rows["estimated_cost_usd"] < 1.0

{'retry_calls': 2, 'fallback_primary_calls': 2, 'limit': 1, 'limited_model_calls': 1, 'estimated_cost_usd': 0.0}
